In [0]:
%sql
create volume if not exists customers_delta


In [0]:
%sql
CREATE TABLE IF NOT EXISTS customers_dimension ( id INT, name STRING, email STRING ) USING DELTA;

Delta Lake supports schema evolution when you use MERGE, UPDATE, or ALTER TABLE with the right options.
ALTER TABLE customers_dimension ADD COLUMNS (phone STRING);
INSERT INTO customers_dimension (id, name, email, phone)
VALUES
  (4, 'Diana', 'diana@example.com', '+91-9876543210');



In [0]:
%sql
-- If duplicates already exist, you can clean them up:
CREATE OR REPLACE TABLE customers_dimension
USING DELTA AS
SELECT id, name, email
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY name) AS rn
  FROM customers_dimension
)
WHERE rn = 1;


In [0]:
%sql
create volume if not exists customers;

In [0]:
df = spark.read.format("CSV").option("header", "true").load("/Volumes/workspace/default/customers/*.txt")
# df.display()

df.write.format("delta")\
.mode("append")\
.option("mergeSchema", "true")\
.save("/Volumes/workspace/default/customers_delta")

#Managed table
# df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("workspace.default.customers_dimension")





In [0]:
df.count()

In [0]:
%python
# -- Unmanaged Table 
# -- sql syntax needs to provide the location abfss:// or s3://
# -- CREATE TABLE workspace.default.customers_dimension
# -- USING DELTA
# -- LOCATION '/Volumes/workspace/default/customers_delta';

df = spark.read.format("delta").load("/Volumes/workspace/default/customers_delta")
df.count()
# df.write.format("delta").saveAsTable("workspace.default.customers_dimension")


In [0]:
%sql
-- drop table workspace.default.customers_dimension

select * from workspace.default.customers_dimension

In [0]:
# Files are processed only once.
# Read Delta table from the Volume path
df = spark.read.format("delta").load("/Volumes/workspace/default/customers_delta")

# Show a few rows
df.show(5,False)
# df.count()


In [0]:
# Read the Delta table properly
df = spark.read.format("delta").load("/Volumes/workspace/default/customers_delta")

# Display the data
display(df)


In [0]:
df=spark.read.json("/Volumes/workspace/default/customers_delta/_delta_log/00000000000000000001.json")
df.display()